In [1]:
KEEP_DRIVER_OPEN = False
SHOW_UI = False
TEST = False

In [2]:
import sys
sys.path.append('../../Storage')  # Updated path for reorganized structure

In [3]:
import datetime

# Get the current date and time
current_datetime = datetime.datetime.now()

# Format and print the current date and time
print("Last session Date and Time:", current_datetime)

Last session Date and Time: 2026-01-25 20:43:14.843168


In [4]:
!pip3 install -r ../requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: '../requirements.txt'


In [5]:
try:
    basestring
except NameError:
    basestring = str

from datetime import datetime
from decimal import Decimal
from future.utils import iteritems
import dateutil.parser

class BaseModel(object):

    """ Base class for other models. """
    
    def __init__(self, **kwargs):
        self._default_params = {}

    @classmethod
    def _NewFromJsonDict(cls, data, **kwargs):
        if kwargs:
            for key, val in kwargs.items():
                data[key] = val
        return cls(**data)

class Book(BaseModel):
    """A class that represents the Bitso orderbook and it's limits"""

    def __init__(self, **kwargs):
        self._default_params = {
            'symbol': kwargs.get('book'),
            'minimum_amount': Decimal(kwargs.get('minimum_amount')),
            'maximum_amount': Decimal(kwargs.get('maximum_amount')),
            'minimum_price': Decimal(kwargs.get('minimum_price')),
            'maximum_price': Decimal(kwargs.get('maximum_price')),
            'minimum_value': Decimal(kwargs.get('minimum_value')),
            'maximum_value': Decimal(kwargs.get('maximum_value'))
        }
        
        for (param, val) in self._default_params.items():
            setattr(self, param, val)

    def __repr__(self):
        return "Book(symbol={symbol})".format(symbol=self.symbol)
    
class AvailableBooks(BaseModel):
    """A class that represents Bitso's orderbooks"""
    def __init__(self, **kwargs):
        self.books = []
        for ob in kwargs.get('payload'):
            self.books.append(ob['book'])
            setattr(self, ob['book'], Book._NewFromJsonDict(ob))

    def __repr__(self):
        return "AvilableBooks(books={books})".format(books=','.join(self.books))


In [6]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

#
#The MIT License (MIT)
#
#Copyright (c) 2016 Mario Romero 
#
#Permission is hereby granted, free of charge, to any person obtaining a copy
#of this software and associated documentation files (the "Software"), to deal
#in the Software without restriction, including without limitation the rights
#to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
#copies of the Software, and to permit persons to whom the Software is
#furnished to do so, subject to the following conditions:
#
#The above copyright notice and this permission notice shall be included in all
#copies or substantial portions of the Software.
#
#THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
#IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
#FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
#AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
#LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
#OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
#SOFTWARE.

from __future__ import absolute_import

import hashlib
import hmac
import json
import time
import requests

from future.utils import iteritems

try:
    from urllib.parse import urlparse, urlencode
except ImportError:
    from urlparse import urlparse
    from urllib import urlencode

def current_milli_time():
    nonce =  str(int(round(time.time() * 1000000)))
    return nonce

class ApiError(Exception):
    pass

class ApiClientError(Exception):
    pass

class Api(object):
    """A python interface for the Bitso API

    Example usage:
      To create an instance of the bitso.Api class, without authentication:
      
        >>> import bitso
        >>> api = bitso.Api()
      
      To get the Bitso price ticker:
      
        >>> ticker = api.ticker()
        >>> print ticker.ask
        >>> print ticker.bid

      To use the private endpoints, initiate bitso.Api with a client_id,
      api_key, and api_secret (see https://bitso.com/developers?shell#private-endpoints):
      
        >>> api = bitso.Api(API_KEY, API_SECRET)
        >>> balance = api.balance()
        >>> print balance.btc_available
        >>> print balance.mxn_available
    """
    
    def __init__(self, key=None, secret=None, timeout=0):
        """Instantiate a bitso.Api object.
        
        Args:
          key:
            Bitso API Key 
          secret:
            Bitso API Secret

  
        """
        self.base_url_v2 = "https://bitso.com/api/v2"
        self.base_url = "https://bitso.com/api/v3"
        self.key = key
        self._secret = secret
        self.timeout = timeout

    def available_books(self):
        """
        Returns:
          A list of bitso.AvilableBook instances
        """
        url = '%s/available_books/' % self.base_url
        resp = self._request_url(url, 'GET')
        return AvailableBooks._NewFromJsonDict(resp)
    
    def _build_auth_payload(self):
        parameters = {}
        parameters['key'] = self.key
        parameters['nonce'] = str(int(time.time()))
        msg_concat = parameters['nonce']+self.client_id+self.key
        parameters['signature'] = hmac.new(self._secret.encode('utf-8'),
                                           msg_concat.encode('utf-8'),
                                           hashlib.sha256).hexdigest()
        return parameters

    def _build_auth_header(self, http_method, url, json_payload=''):
        if json_payload == {} or json_payload=='{}':
            json_payload = ''
        url_components = urlparse(url)
        request_path = url_components.path
        if url_components.query != '':
            request_path+='?'+url_components.query
        nonce = current_milli_time()
        msg_concat = nonce+http_method.upper()+request_path+json_payload
        signature = hmac.new(self._secret.encode('utf-8'),
                                 msg_concat.encode('utf-8'),
                                 hashlib.sha256).hexdigest()
        return {'Authorization': 'Bitso %s:%s:%s' % (self.key, nonce, signature)}

    
    def _request_url(self, url, verb, params=None, private=False):
        headers=None
        if params == None:
            params = {}
        params = {k: v.decode("utf-8") if isinstance(v, bytes) else v for k, v in params.items()}
        if private:
            headers = self._build_auth_header(verb, url, json.dumps(params))
        if verb == 'GET':
            url = self._build_url(url, params)
            if private:
                headers = self._build_auth_header(verb, url)
            try:
                resp = requests.get(url, headers=headers, timeout=self.timeout)
            except requests.RequestException as e:
                raise
        elif verb == 'POST':
            try:
                resp = requests.post(url, json=params, headers=headers, timeout=self.timeout)
            except requests.RequestException as e:
                raise
        elif verb == 'DELETE':
            try:
                resp = requests.delete(url, headers=headers, timeout=self.timeout)
            except requests.RequestException as e:
                raise
        content = resp.content
        data = self._parse_json(content if isinstance(content, basestring) else content.decode('utf-8'))
        return data

    def _build_url(self, url, params):
        if params and len(params) > 0:
            url = url+'?'+self._encode_parameters(params)
        return url

    def _encode_parameters(self, parameters):
        if parameters is None:
            return None
        else:
            param_tuples = []
            for k,v in parameters.items():
                if v is None:
                    continue
                if isinstance(v, (list, tuple)):
                    for single_v in v:
                        param_tuples.append((k, single_v))
                else:
                    param_tuples.append((k,v))
            return urlencode(param_tuples)


         
    def _parse_json(self, json_data):
        try:
            data = json.loads(json_data)
            self._check_for_api_error(data)
        except:
            raise
        return data

    def _check_for_api_error(self, data):
        if data['success'] != True:
            raise ApiError(data['error'])
        if 'error' in data:
            raise ApiError(data['error'])
        if isinstance(data, (list, tuple)) and len(data)>0:
            if 'error' in data[0]:
                raise ApiError(data[0]['error'])

In [7]:
api = Api(timeout=5)
avb_books = api.available_books()
print(f"Total Available Books: {len(avb_books.books)}")
print(f"Available Books: {avb_books.books}")

Total Available Books: 99
Available Books: ['ada_usd', 'sol_mxn', 'dydx_usd', 'eth_btc', 'tusd_btc', 'ondo_usd', 'ltc_mxn', 'usd_ars', 'eth_ars', 'crv_usd', 'yfi_usd', 'sol_brl', 'trx_usd', 'xlm_usd', 'btc_usd', 'snx_usd', 'avax_mxn', 'ldo_usd', 'xrp_usdt', 'paxg_usd', 'btc_usdt', 's_usd', 'btc_usds', 'omg_usd', 'pyusd_mxn', 'popcat_usd', 'psg_usd', 'pepe_usd', 'bar_usd', 'sol_usdt', 'avax_usd', 'eth_usd', 'uni_usd', 'sushi_usd', 'bonk_usd', 'axs_usd', 'hbar_usd', 'bat_usd', 'eur_usd', 'wif_usd', 'comp_usd', 'render_usd', 'ton_usd', 'usd_cop', 'floki_usd', 'bch_usd', 'fet_usd', 'dot_usd', 'mana_mxn', 'near_usd', 'btc_mxn', 'usdt_brl', 'eth_usdt', 'grt_usd', 'sand_usd', 'ltc_usd', 'bal_usd', 'brl1_brl', 'usdt_mxn', 'chz_usd', 'btc_brl', 'shib_usd', 'sol_usd', 'atom_usd', 'enj_usd', 'hype_usd', 'usd_usdt', 'lrc_usd', 'aave_usd', 'sky_usd', 'gala_usd', 'xrp_usd', 'doge_usd', 'virtual_usd', 'trx_mxn', 'btc_ars', 'bat_mxn', 'tigres_usd', 'usd_brl', 'eur_mxn', 'ape_usd', 'usd_mxn', 'arb_usd'

In [8]:
usd_books = [book for book in avb_books.books if 'mxn' not in book]
usd_books = [book for book in usd_books if 'brl' not in book]
usd_books = [book for book in usd_books if 'cop' not in book]
usd_books = [book for book in usd_books if 'ars' not in book]
print(f"Total USD Available Books: {len(usd_books)}")
print(f"USD Available Books: {usd_books}")

Total USD Available Books: 69
USD Available Books: ['ada_usd', 'dydx_usd', 'eth_btc', 'tusd_btc', 'ondo_usd', 'crv_usd', 'yfi_usd', 'trx_usd', 'xlm_usd', 'btc_usd', 'snx_usd', 'ldo_usd', 'xrp_usdt', 'paxg_usd', 'btc_usdt', 's_usd', 'btc_usds', 'omg_usd', 'popcat_usd', 'psg_usd', 'pepe_usd', 'bar_usd', 'sol_usdt', 'avax_usd', 'eth_usd', 'uni_usd', 'sushi_usd', 'bonk_usd', 'axs_usd', 'hbar_usd', 'bat_usd', 'eur_usd', 'wif_usd', 'comp_usd', 'render_usd', 'ton_usd', 'floki_usd', 'bch_usd', 'fet_usd', 'dot_usd', 'near_usd', 'eth_usdt', 'grt_usd', 'sand_usd', 'ltc_usd', 'bal_usd', 'chz_usd', 'shib_usd', 'sol_usd', 'atom_usd', 'enj_usd', 'hype_usd', 'usd_usdt', 'lrc_usd', 'aave_usd', 'sky_usd', 'gala_usd', 'xrp_usd', 'doge_usd', 'virtual_usd', 'tigres_usd', 'ape_usd', 'arb_usd', 'pol_usd', 'qnt_usd', 'mana_usd', 'algo_usd', 'neiro_usd', 'link_usd']


In [9]:
usd_books = [book.replace('_', '-') for book in usd_books]
print(f"USD Available Books: {usd_books}")

USD Available Books: ['ada-usd', 'dydx-usd', 'eth-btc', 'tusd-btc', 'ondo-usd', 'crv-usd', 'yfi-usd', 'trx-usd', 'xlm-usd', 'btc-usd', 'snx-usd', 'ldo-usd', 'xrp-usdt', 'paxg-usd', 'btc-usdt', 's-usd', 'btc-usds', 'omg-usd', 'popcat-usd', 'psg-usd', 'pepe-usd', 'bar-usd', 'sol-usdt', 'avax-usd', 'eth-usd', 'uni-usd', 'sushi-usd', 'bonk-usd', 'axs-usd', 'hbar-usd', 'bat-usd', 'eur-usd', 'wif-usd', 'comp-usd', 'render-usd', 'ton-usd', 'floki-usd', 'bch-usd', 'fet-usd', 'dot-usd', 'near-usd', 'eth-usdt', 'grt-usd', 'sand-usd', 'ltc-usd', 'bal-usd', 'chz-usd', 'shib-usd', 'sol-usd', 'atom-usd', 'enj-usd', 'hype-usd', 'usd-usdt', 'lrc-usd', 'aave-usd', 'sky-usd', 'gala-usd', 'xrp-usd', 'doge-usd', 'virtual-usd', 'tigres-usd', 'ape-usd', 'arb-usd', 'pol-usd', 'qnt-usd', 'mana-usd', 'algo-usd', 'neiro-usd', 'link-usd']


In [10]:
def from_book(book):
    cum = []
    start = False
    for usd_book in usd_books:
        if usd_book == book:
            start = True
        if start:
            cum.append(usd_book)
    print(f"From chosen USD Available Book: {cum}")
    return cum

In [11]:
import pgConn
import PostgresSQL_table_queries

# Table schema includes UNIQUE(book, date); pg_conn.save_to_postgres uses ON CONFLICT DO NOTHING to avoid duplicates
pg_conn = pgConn.PgConn(tablename="historical", dbname="cryptostocks", user="postgres")
pg_conn.init_db(PostgresSQL_table_queries.HISTORICAL_CRYPTO_STOCKS_TABLE_QUERY)
# pg_conn.set_table("another_custom_table_name")
# pg_conn.save_to_postgres(row_data, header)
# # Perform other operations using pg_conn
# pg_connpg_conn = super.initDB('postgres', "historical", "cryptostocks", "postgres", PostgresSQL_table_queries.HISTORICAL_CRYPTO_STOCKS_TABLE_QUERY).close_connection()

Connection to the database successful!
Table name set to: historical
Table 'historical' already exists.


<connection object at 0x114b0ace0; dsn: 'user=postgres password=xxx dbname=cryptostocks host=localhost port=5432', closed: 0>

In [12]:
import CloudStorage as cs
import boto3
import os

def store_to_s3(bucket_name, folder_name):
    # Bucket name and folder paths
    local_file_path = "path/to/local/file.txt"
    #s3_file_path = f"{folder_name}/file.txt"
    current_directory = os.getcwd()
    s3_file_path = f"{current_directory}/{folder_name}/file.txt"
    
    # Create the bucket and folder if they don't exist
    cs.create_bucket(bucket_name)
    s3 = boto3.resource('s3')
    bucket = s3.Bucket(bucket_name)
    bucket.put_object(Key=s3_file_path, Body="")  # Create an empty object to create the folder

    # Upload the file to S3
    cs.upload_file_to_s3(bucket_name, local_file_path, s3_file_path)

In [13]:
import time
import json
import time
import pandas as pd
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.firefox.service import Service as FirefoxService
from webdriver_manager.firefox import GeckoDriverManager
from selenium.webdriver.common.by import By

In [14]:
contents = []
url = f'https://finance.yahoo.com/lookup'
xpath = "/html/body/div[1]/div/div/div[1]/div/div[3]/div[1]/div/div[2]/div/div/div/ul/li[1]/div/div/div[2]/h3/a"

In [15]:
MAIN_SECTION_STOCK_DATA_HTML_EL = "/html/body/div[1]/main/section/section/section"
HISTORIAL_DATA_BTN = "/html/body/div[1]/div/div/div[1]/div/div[2]/div/div/div[7]/div/div/section/div/ul/li[4]/a"

STOCKS_HTML_TABLE = "/html/body/div[1]/main/section/section/section/article/div[1]/div[3]/table"
# Try multiple XPath options as Yahoo Finance page structure can vary
STOCKS_HTML_TABLE_BODY_OPTIONS = [
    "/html/body/div[1]/main/section/section/section/article/div[1]/div[3]/table/tbody",  # Primary option (from WebScraper.py)
    "/html/body/div[1]/main/section/section/section/article/div[1]/div[3]/table/tbody",  # Same as primary
    "/html/body/div[2]/main/section/section/section/article/div[1]/div[3]/table/tbody",  # Alternative div structure
    "/html/body/div[2]/div[2]/main/section/section/section/section/div[1]/div[3]/table/tbody",  # Alternative option from error
    "//table[@data-test='historical-prices']/tbody",  # Using data attribute
    "//table[contains(@class, 'W(100%)')]/tbody",  # Using class attribute
    "//table[contains(@class, 'table')]/tbody",  # Generic table class
    "//tbody[.//tr]",  # Any tbody with rows
]
STOCKS_HTML_TABLE_BODY = STOCKS_HTML_TABLE_BODY_OPTIONS[0]  # Default to first option
NO_RESULTS_FOUND_HTML_SPAN_EL = "/html/body/div[1]/div/div/div[1]/div/div[3]/div[1]/div/div[1]/div/div/section/section/div/div/span/span"
NO_RESULTS_FOUND_HTML_DIV_EL = "/html/body/div[2]/main/section/section/section/article/section/div[3]"
NO_RESULTS_FOUND_HTML_CLASS_EL = "noData"

NO_MATCH_RESULTS_URL = "https://finance.yahoo.com/lookup/?s="

In [16]:
from datetime import datetime

def parse_date(date_str):
    # Convert the month name to a numerical representation using a dictionary
    month_dict = {
        "Jan": "01",
        "Feb": "02",
        "Mar": "03",
        "Apr": "04",
        "May": "05",
        "Jun": "06",
        "Jul": "07",
        "Aug": "08",
        "Sep": "09",
        "Oct": "10",
        "Nov": "11",
        "Dec": "12",
    }

    date_str = date_str.replace(",", "")

    # Split the date string into month, day, and year
    month, day, year = date_str.split()

    # Get the numerical representation of the month from the dictionary
    month_number = month_dict[month]

    # Create a new date string in the format 'year-month-day' (e.g., '2023-08-01')
    formatted_date_str = f"{year}-{month_number}-{day}"

    # Parse the formatted date string to a datetime object
    parsed_date = datetime.strptime(formatted_date_str, "%Y-%m-%d")

    return parsed_date

def parse_row_data(row_data):
    try:
        date_format = '%Y-%m-%d'  # Format for parsing date strings

        # Remove commas from numeric values
        row_data = [item.replace(",", "") if isinstance(item, str) else item for item in row_data]

        # Parse elements at specific positions into desired data types
        row_data[0] = parse_date(row_data[0])
        row_data[1] = float(row_data[1])
        row_data[2] = float(row_data[2])
        row_data[3] = float(row_data[3])
        row_data[4] = float(row_data[4])
        row_data[5] = float(row_data[5])
        row_data[6] = int(row_data[6])
        return row_data
    except Exception as e:
        print("error during parsing data:", e, "row_data: ", row_data)

In [17]:
import csv
import os

def save_unavailable_book(book_name):
    try:
        current_directory = os.getcwd()
        unavailable_books_file = os.path.join(current_directory, "unavailable_books.csv")
        
        file_exists = os.path.isfile(unavailable_books_file)
        with open(unavailable_books_file, "a", newline="") as csvfile:
            writer = csv.writer(csvfile)
            if not file_exists:
                writer.writerow(["book"])  # Add header if the file is newly created
            writer.writerow([book_name])
        print(f"Book '{book_name}' added to unavailable_books.csv")
    except Exception as e:
        print(f"Error while saving book '{book_name}' to CSV: {e}")

In [18]:
def load_page_with_timeout(driver, url, timeout=10):
    """
    Load a web page with a timeout. If the page does not load within the timeout, stop the loading process.

    :param driver: Selenium WebDriver instance.
    :param url: The URL of the page to load.
    :param timeout: Maximum time to wait for the page to load, in seconds.
    """
    try:
        # Start loading the page
        driver.get(url)

        # Wait for the document.readyState to be "complete" within the timeout
        WebDriverWait(driver, timeout).until(
            lambda d: d.execute_script("return document.readyState") == "complete"
        )
        
        # Additional wait for jQuery (if present) and any dynamic content
        try:
            WebDriverWait(driver, 2).until(
                lambda d: d.execute_script("return typeof jQuery === 'undefined' || jQuery.active == 0")
            )
        except:
            pass  # jQuery might not be present, that's okay
        
        print(f"Page loaded successfully within {timeout} seconds.")
    except TimeoutException:
        # Stop the loading process if timeout is reached
        print(f"Page did not load within {timeout} seconds. Cancelling load...")
        driver.execute_script("window.stop()")  # Cancel the page loading

def disable_auto_refresh(driver, url, timeout=5):
    """
    Load a web page and disable auto-refresh behavior.
    
    :param driver: Selenium WebDriver instance.
    :param url: The URL of the page to load.
    :param timeout: Maximum time to wait for the page to load, in seconds.
    """
    driver.get(url)
    
    try:
        # Wait for the page to load completely
        WebDriverWait(driver, timeout).until(
            lambda d: d.execute_script("return document.readyState") == "complete"
        )
        print("Page loaded successfully.")

        # Disable auto-refresh caused by meta tags
        driver.execute_script("""
            const metaTags = document.querySelectorAll('meta[http-equiv="refresh"]');
            metaTags.forEach(tag => tag.remove());
        """)

        # Cancel any JavaScript-based auto-refresh mechanisms
        driver.execute_script("""
            const cancelAutoRefresh = () => {
                const originalSetInterval = window.setInterval;
                const originalSetTimeout = window.setTimeout;
                
                // Override setInterval and setTimeout to prevent refresh
                window.setInterval = (...args) => {
                    if (args[0].toString().includes('location.reload') || 
                        args[0].toString().includes('window.location')) {
                        console.log('Blocked setInterval auto-refresh.');
                        return null;
                    }
                    return originalSetInterval(...args);
                };
                
                window.setTimeout = (...args) => {
                    if (args[0].toString().includes('location.reload') || 
                        args[0].toString().includes('window.location')) {
                        console.log('Blocked setTimeout auto-refresh.');
                        return null;
                    }
                    return originalSetTimeout(...args);
                };
            };
            cancelAutoRefresh();
        """)

        print("Auto-refresh has been disabled.")
    except Exception as e:
        print(f"Error: {e}")

In [19]:
def get_dynamic_url(ticker, period1=1410825600, period2=1690675200, interval="1d",adjclose="true"):
    return f'https://finance.yahoo.com/quote/{ticker.upper()}/history?period1={period1}&period2={period2}&interval={interval}&filter=history&frequency={interval}&includeAdjustedClose={adjclose}'

def scroll_to_bottom(driver):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
    
def is_at_bottom(driver):
    lastHeight = driver.execute_script("return document.documentElement.scrollHeight")
    while True:
        driver.execute_script("var scrollingElement = (document.scrollingElement || document.body);scrollingElement.scrollTop = scrollingElement.scrollHeight;")
        height = driver.execute_script("return document.documentElement.scrollHeight")
        driver.execute_script("window.scrollTo(0, " + str(height) + ");")
        time.sleep(2)
        if lastHeight == height:
            print("scrolling down task finished")
            break
        lastHeight = height

def printInnerHTML(xpath):
    # Get the inner HTML of the specific element
    specific_element = driver.find_element(By.XPATH, xpath)
    
    inner_html = specific_element.get_attribute('innerHTML')

    # Print the inner HTML of the specific element
    print(inner_html)


def printXPathAndClass(el):
    # Get the XPath of the element
    element_xpath = el.get_attribute("xpath")
    print("Element XPath:", element_xpath)
    
    # Get the class attribute of the element
    element_class = el.get_attribute("class")
    print("Element Class:", element_class)
        
def check_tab_header(driver):
    try:
        element = driver.find_element(By.XPATH, '//*[@id="quote-nav"]')
        #tab = driver.find_element(By.XPATH, '/html/body/div[1]/div/div/div[1]/div/div[2]/div/div/div[7]/section/div/ul/li[3]/a')
        return True
    except Exception as e:
        print(f"financial header or historical data tab does not exist: {e}")
        return False

def check_html_el_exist(driver, html_el, selector = 'xpath'):
    wait = WebDriverWait(driver, 3.0)
    try:
        if selector == 'xpath':
            wait.until(EC.presence_of_element_located((By.XPATH, html_el)))
        elif selector == 'class':
            wait.until(EC.presence_of_element_located((By.CLASS_NAME, html_el)))
        return True
    except NoSuchElementException:
        print("Element does not exist")
        return False

def notmatchresult_test_conditions(driver):
    try:
        print(f"current driver url: {driver.current_url}")
    except NoSuchElementException:
        print("current driver url doesnt' match")
    try:
        print(f"checking if html_el_exists by xpath: {check_html_el_exist(driver, NO_RESULTS_FOUND_HTML_SPAN_EL, 'xpath')}")
    except Exception as e:
        print("'no results found' HTML element by XPATH doesn't exists:", e)
    try:
        print(f"checking if html_el_exists by class: {check_html_el_exist(driver, NO_RESULTS_FOUND_HTML_CLASS_EL, 'class')}")
    except Exception as e:
        print("'no results found' HTML element by CLASS_NAME doesn't exists:", e)
    
def nomatchresult(driver, book):
    if TEST == True:
        notmatchresult_test_conditions(driver)
    try:
        if (driver.current_url == f"{NO_MATCH_RESULTS_URL}{book.upper()}" or check_html_el_exist(driver, NO_RESULTS_FOUND_HTML_SPAN_EL, 'xpath') or check_html_el_exist(driver, NO_RESULTS_FOUND_HTML_CLASS_EL, 'class')):
            print(f"no data was found for {book.upper()}")
            return True
    except Exception as nse:
            print(f"{book.upper()} book was found!")
            return False
            
def lookup_ticker(driver, ticker):
    RejectAll= driver.find_element(By.XPATH, '/html/body/div[1]/div/div/div[1]/div/div[3]/div[2]/div/div/div/div/div/div[1]/div/div/div/form/input')
    action = ActionChains(driver)
    action.click(on_element = RejectAll)
    action.perform()
    time.sleep(5)
    SearchBar = driver.find_element(By.ID, "yfin-usr-qry")
    SearchBar.send_keys(ticker.upper())
    SearchBar.send_keys(Keys.ENTER)

def select_historical_li(driver):
    li_historical_a = driver.find_element(By.XPATH, '/html/body/div[1]/div/div/div[1]/div/div[2]/div/div/div[7]/div/div/section/div/ul/li[4]/a')
    action = ActionChains(driver)
    action.click(on_element = li_historical_a)
    action.perform()
    time.sleep(3)

def disable_ad(driver): 
    wait = WebDriverWait(driver, 3.0)
    try:
        ad_element = '//*[@id="Col1-0-Ad-Proxy"]'
        wait.until(EC.presence_of_element_located((By.XPATH, ad_element)))
        driver.execute_script("arguments[0].style.display = 'none';", ad_element)
    except Exception as e:
        print("ad element was not found")

def historical_stock_search_selector(driver):
    print("selecting historical dropdown menu")
    wait = WebDriverWait(driver, 3.0)
    try:
        selector1 = "/html/body/div[1]/main/section/section/section/article/div[1]/div[1]/div[1]" # Menu container
        wait.until(EC.presence_of_element_located((By.XPATH, selector1)))
        return selector1
    except Exception as e:
        print("selector1 for time period not found trying the second")
        printInnerHTML("/html/body/div[1]/main/section/section/section/article/")
        try:
            selector2 = "/html/body/div[1]/div/div/div[1]/div/div[3]/div[1]/div/div[2]/div/div/section"
            wait.until(EC.presence_of_element_located((By.XPATH, selector2)))
            return selector2
        except Exception as e:
            print("selector2 for time period not found")
            
def select_historical(driver, time_period, freq):
    print("Assessing historical stock prices table data ...", end='', flush=True)
    # disable_ad(driver)
    wait = WebDriverWait(driver, 3.0)
    hs_se = historical_stock_search_selector(driver)
    action = ActionChains(driver)
    hs_se_button = driver.find_element(By.XPATH, f"{hs_se}/button")
    hs_se_button.click()

    try:
        '''
        TODO Add Frequency HTML button element
        '''
        hs_period_dropdown_div = ''
        if (time_period == '1d'):
            wait.until(EC.presence_of_element_located((By.XPATH, f"{hs_se}/div/div/div[2]/section/div[1]/button[1]")))
            hs_period_dropdown_div = driver.find_element(By.XPATH, f"{hs_se}/div/div/div[2]/section/div[1]/button[1]")
            hs_period_dropdown_div.click()
        elif (time_period == '5d'):
            wait.until(EC.presence_of_element_located((By.XPATH, f"{hs_se}/div/div/div[2]/section/div[1]/button[2]")))
            hs_period_dropdown_div = driver.find_element(By.XPATH, f"{hs_se}/div/div/div[2]/section/div[1]/button[2]")
            hs_period_dropdown_div.click()
        elif (time_period == '1y'):
            wait.until(EC.presence_of_element_located((By.XPATH, f"{hs_se}/div/div/div[2]/section/div[1]/button[6]")))
            hs_period_dropdown_div = driver.find_element(By.XPATH, f"{hs_se}/div/div/div[2]/section/div[1]/button[6]")
            hs_period_dropdown_div.click()
    except Exception as e:
        print("Error on select_historical(): ", e)
        print("====================================================================")
        print("Printing inner HTML")
        print("====================================================================")
        printInnerHTML(hs_se)
    
    
    wait.until(EC.presence_of_element_located((By.XPATH, STOCKS_HTML_TABLE)))
    print("Task finished")

def find_table_body_with_fallback(driver, timeout=10):
    """
    Try to find the table body element using multiple XPath options.
    Returns the table element if found, None otherwise.
    """
    wait = WebDriverWait(driver, timeout)
    
    # First, wait for the table itself to be present and visible (using any of the possible table XPaths)
    table_xpaths = [
        STOCKS_HTML_TABLE,
        "/html/body/div[1]/main/section/section/section/article/div[1]/div[3]/table",
        "/html/body/div[2]/main/section/section/section/article/div[1]/div[3]/table",
        "//table[@data-test='historical-prices']",
        "//table[contains(@class, 'W(100%)')]",
        "//table[contains(@class, 'table')]",
    ]
    
    table_element = None
    for table_xpath in table_xpaths:
        try:
            # Wait for both presence and visibility
            table_element = wait.until(EC.visibility_of_element_located((By.XPATH, table_xpath)))
            print(f"Table found using XPath: {table_xpath}")
            break
        except TimeoutException:
            continue
    
    if table_element is None:
        print("ERROR: Could not find table element with any XPath")
        return None
    
    # Wait a bit more for tbody to be populated (sometimes it loads after the table)
    time.sleep(1)
    
    # Now try to find the tbody within the found table
    try:
        # Try to find tbody directly within the table
        tbody_wait = WebDriverWait(table_element, 5)
        tbody = tbody_wait.until(lambda d: table_element.find_element(By.TAG_NAME, "tbody"))
        # Check if tbody has rows
        rows = tbody.find_elements(By.TAG_NAME, "tr")
        if len(rows) > 0:
            print(f"Table body found using TAG_NAME within table (found {len(rows)} rows)")
            return tbody
    except (NoSuchElementException, TimeoutException) as e:
        print(f"Could not find tbody within table: {e}")
    
    # If that fails, try the full XPath options
    for tbody_xpath in STOCKS_HTML_TABLE_BODY_OPTIONS:
        try:
            tbody = wait.until(EC.visibility_of_element_located((By.XPATH, tbody_xpath)))
            # Verify it has rows
            rows = tbody.find_elements(By.TAG_NAME, "tr")
            if len(rows) > 0:
                print(f"Table body found using XPath: {tbody_xpath} (found {len(rows)} rows)")
                return tbody
        except (TimeoutException, NoSuchElementException):
            continue
    
    print("ERROR: Could not find table body element with any XPath or table body has no rows")
    return None

In [20]:
DRIVER_PATH = "/chromedriver/chromedriver"
options = webdriver.ChromeOptions()
options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.5735.90 Safari/537.36")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-extensions")
options.add_argument("--proxy-server='direct://'")
options.add_argument("--proxy-bypass-list=*")
options.add_argument("--start-maximized")
if not SHOW_UI:
    options.add_argument('--headless')
options.add_argument('--disable-gpu')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--no-sandbox')
options.add_argument('--ignore-certificate-errors')

In [21]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)

In [22]:
REFERENCE = 'https://finance.yahoo.com'
Header = ["reference", "book", "date", "open", "high", "low", "close", "adj_close", "volume"]
n = len(Header)
Debug = False
time_period = '1d'
frequency = 'daily'
show_row_data = True
frombook = 'virtual-usd'
if (len(frombook) > 0):
    usd_books = from_book(frombook)

try:
    WebDriverWait(driver,5).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    try:
        for book in usd_books:
            print(f'Book: {book}')
            target_url = f"https://finance.yahoo.com/quote/{book.upper()}/history?p={book.upper()}"
            print(target_url)
            load_page_with_timeout(driver, target_url, timeout=10)
            
            if(nomatchresult(driver, book)):
                print("skipping to next ticket")
                # save_unavailable_book(book)
                print("====================================================================")
                continue
                
            #select_historical(driver, time_period, frequency)
            time.sleep(2)  # Give page more time to load

            #is_at_bottom(driver)
            # Use the new function with fallback XPath options
            table = find_table_body_with_fallback(driver, timeout=10)
            
            if table is None:
                print(f"ERROR: Could not find table for {book}. Skipping to next book.")
                print("====================================================================")
                continue
            
            # Get all rows of the table
            rows = table.find_elements(By.TAG_NAME, "tr")

            # Create an empty list to store the table data
            #table_data = []
            #df_book = pd.DataFrame(table_data, columns=Header)
            # Iterate through each row
            print("Scraping raw stock prices data task started")
            for idx, row in enumerate(rows):
                if time_period == '1d' and idx > 63:
                    break
                # Get all columns (cells) of the row
                columns = row.find_elements(By.TAG_NAME, "td")
                row_data = []
                row_data = [column.text for column in columns if column.text != '-']
                if(len(row_data) != 7):
                    print("skipping to next row")
                    continue
                row_data = parse_row_data(row_data)
                row_data.insert(0, book)
                row_data.insert(0, REFERENCE)
                try:
                    if (show_row_data):
                        print("test: ", row_data)
                    pg_conn.save_to_postgres(row_data, Header)
                except Exception as e:
                    print(f"error while saving to postgres: {e}")
                #df_book = pd.concat([df_book, pd.DataFrame([row_data], columns=Header)], ignore_index=True)
            #df = pd.concat([df, df_book], ignore_index=True)
            #num_rows, num_columns = df.shape
            #last_five_rows = df.tail(3)
            print("Scraping raw stock prices data task finished")
            print("====================================================================")
        print("**All book data was scraped**")
    except NoSuchElementException as nse:
        print(nse)
        print("-----")
        print(str(nse))
        print("-----")
        print(nse.args)
        print("=====")
except TimeoutException as toe:
    print(toe)
    print("-----")
    print(str(toe))
    print("-----")
    print(toe.args)
finally:
    if(Debug):
        delete_table("historical", conn)
    pg_conn.close_connection()
if (not KEEP_DRIVER_OPEN):
    driver.close()

From chosen USD Available Book: ['virtual-usd', 'tigres-usd', 'ape-usd', 'arb-usd', 'pol-usd', 'qnt-usd', 'mana-usd', 'algo-usd', 'neiro-usd', 'link-usd']
Book: virtual-usd
https://finance.yahoo.com/quote/VIRTUAL-USD/history?p=VIRTUAL-USD
Page did not load within 10 seconds. Cancelling load...
VIRTUAL-USD book was found!
Table found using XPath: //table[contains(@class, 'table')]
Table body found using TAG_NAME within table (found 365 rows)
Scraping raw stock prices data task started
test:  ['https://finance.yahoo.com', 'virtual-usd', datetime.datetime(2026, 1, 26, 0, 0), 0.765279, 0.795766, 0.763955, 0.794509, 0.794509, 69044688]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'virtual-usd', datetime.datetime(2026, 1, 24, 0, 0), 0.829164, 0.837326, 0.814897, 0.822948, 0.822948, 38157847]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'virtual-usd', datetime.datetime(2026, 1, 23, 0, 0), 0.837336, 0.877882, 0.82643, 0.829178, 0.829178, 88886839]
Savi

Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'virtual-usd', datetime.datetime(2025, 12, 12, 0, 0), 0.840579, 0.853839, 0.780748, 0.79955, 0.79955, 70155161]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'virtual-usd', datetime.datetime(2025, 12, 11, 0, 0), 0.842588, 0.855826, 0.795236, 0.840663, 0.840663, 90512463]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'virtual-usd', datetime.datetime(2025, 12, 10, 0, 0), 0.885539, 0.904365, 0.842414, 0.842571, 0.842571, 93575315]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'virtual-usd', datetime.datetime(2025, 12, 9, 0, 0), 0.834112, 0.918604, 0.832836, 0.885588, 0.885588, 110746195]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'virtual-usd', datetime.datetime(2025, 12, 8, 0, 0), 0.828512, 0.874603, 0.827069, 0.834111, 0.834111, 127387069]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'virtual-usd', datetime.datetime(202

Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'tigres-usd', datetime.datetime(2025, 12, 30, 0, 0), 0.055, 0.0553, 0.0536, 0.0553, 0.0553, 327]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'tigres-usd', datetime.datetime(2025, 12, 29, 0, 0), 0.0534, 0.056, 0.0534, 0.055, 0.055, 517]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'tigres-usd', datetime.datetime(2025, 12, 28, 0, 0), 0.0561, 0.0561, 0.0534, 0.0534, 0.0534, 1480]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'tigres-usd', datetime.datetime(2025, 12, 27, 0, 0), 0.0561, 0.0561, 0.0551, 0.0561, 0.0561, 434]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'tigres-usd', datetime.datetime(2025, 12, 26, 0, 0), 0.0584, 0.059, 0.0554, 0.0561, 0.0561, 456]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'tigres-usd', datetime.datetime(2025, 12, 25, 0, 0), 0.059, 0.0598, 0.0533, 0.0584, 0.0584, 2333]
Saving to postgres d

Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'ape-usd', datetime.datetime(2026, 1, 15, 0, 0), 0.223976, 0.226919, 0.211897, 0.213901, 0.213901, 7282]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'ape-usd', datetime.datetime(2026, 1, 14, 0, 0), 0.236852, 0.236852, 0.221934, 0.223976, 0.223976, 17452]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'ape-usd', datetime.datetime(2026, 1, 13, 0, 0), 0.202783, 0.251885, 0.201788, 0.236852, 0.236852, 16697]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'ape-usd', datetime.datetime(2026, 1, 12, 0, 0), 0.207699, 0.212715, 0.202745, 0.202783, 0.202783, 4987]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'ape-usd', datetime.datetime(2026, 1, 11, 0, 0), 0.210725, 0.213743, 0.205704, 0.207699, 0.207699, 3770]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'ape-usd', datetime.datetime(2026, 1, 10, 0, 0), 0.214719, 0.21571, 0.209719,

Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'ape-usd', datetime.datetime(2025, 11, 27, 0, 0), 0.279986, 0.28502, 0.271994, 0.278986, 0.278986, 6427]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'ape-usd', datetime.datetime(2025, 11, 26, 0, 0), 0.287884, 0.28792, 0.269912, 0.279986, 0.279986, 9227]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'ape-usd', datetime.datetime(2025, 11, 25, 0, 0), 0.28591, 0.287918, 0.269981, 0.287884, 0.287884, 10271]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'ape-usd', datetime.datetime(2025, 11, 24, 0, 0), 0.282882, 0.291021, 0.27886, 0.28591, 0.28591, 10329]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'ape-usd', datetime.datetime(2025, 11, 23, 0, 0), 0.278835, 0.287935, 0.276857, 0.282882, 0.282882, 6279]
Saving to postgres db...done
Scraping raw stock prices data task finished
Book: arb-usd
https://finance.yahoo.com/quote/ARB-USD/history?p=ARB-US

...

Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'mana-usd', datetime.datetime(2025, 11, 27, 0, 0), 0.168894, 0.172736, 0.167596, 0.170668, 0.170668, 20789744]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'mana-usd', datetime.datetime(2025, 11, 26, 0, 0), 0.168887, 0.17012, 0.162521, 0.168894, 0.168894, 24327942]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'mana-usd', datetime.datetime(2025, 11, 25, 0, 0), 0.171203, 0.171617, 0.165297, 0.168887, 0.168887, 24489049]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'mana-usd', datetime.datetime(2025, 11, 24, 0, 0), 0.163644, 0.172245, 0.162478, 0.171196, 0.171196, 27261570]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'mana-usd', datetime.datetime(2025, 11, 23, 0, 0), 0.163833, 0.167964, 0.163294, 0.163644, 0.163644, 22294540]
Saving to postgres db...done
Scraping raw stock prices data task finished
Book: algo-usd
https://finance.yahoo.com/q

Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'algo-usd', datetime.datetime(2025, 12, 16, 0, 0), 0.115471, 0.118041, 0.113207, 0.116862, 0.116862, 40502051]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'algo-usd', datetime.datetime(2025, 12, 15, 0, 0), 0.118999, 0.121801, 0.111814, 0.115471, 0.115471, 51615111]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'algo-usd', datetime.datetime(2025, 12, 14, 0, 0), 0.122031, 0.123465, 0.118328, 0.118999, 0.118999, 36939302]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'algo-usd', datetime.datetime(2025, 12, 13, 0, 0), 0.122938, 0.123185, 0.120809, 0.122031, 0.122031, 32003782]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'algo-usd', datetime.datetime(2025, 12, 12, 0, 0), 0.129938, 0.131096, 0.121884, 0.122938, 0.122938, 52921412]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'algo-usd', datetime.datetime(2025, 12, 11, 0, 0)

Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'neiro-usd', datetime.datetime(2025, 10, 15, 0, 0), 0.005031, 0.005052, 0.004451, 0.004681, 0.004681, 1112954]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'neiro-usd', datetime.datetime(2025, 10, 14, 0, 0), 0.005212, 0.005228, 0.00455, 0.005031, 0.005031, 1219648]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'neiro-usd', datetime.datetime(2025, 10, 13, 0, 0), 0.00515, 0.005397, 0.005012, 0.005212, 0.005212, 1060623]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'neiro-usd', datetime.datetime(2025, 10, 12, 0, 0), 0.004389, 0.005894, 0.004199, 0.00515, 0.00515, 2057635]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'neiro-usd', datetime.datetime(2025, 10, 11, 0, 0), 0.003789, 0.004587, 0.003478, 0.004394, 0.004394, 3003928]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'neiro-usd', datetime.datetime(2025, 10, 10, 0, 0), 0

Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'link-usd', datetime.datetime(2026, 1, 24, 0, 0), 12.21, 12.26, 12.13, 12.19, 12.19, 180740178]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'link-usd', datetime.datetime(2026, 1, 23, 0, 0), 12.24, 12.44, 12.06, 12.21, 12.21, 281712703]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'link-usd', datetime.datetime(2026, 1, 22, 0, 0), 12.39, 12.56, 12.13, 12.24, 12.24, 330622925]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'link-usd', datetime.datetime(2026, 1, 21, 0, 0), 12.11, 12.64, 11.94, 12.39, 12.39, 509903013]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'link-usd', datetime.datetime(2026, 1, 20, 0, 0), 12.88, 12.89, 12.11, 12.11, 12.11, 440295539]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'link-usd', datetime.datetime(2026, 1, 19, 0, 0), 13.33, 13.33, 12.73, 12.88, 12.88, 579914287]
Saving to postgres db...done

Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'link-usd', datetime.datetime(2025, 12, 4, 0, 0), 14.68, 14.91, 14.01, 14.25, 14.25, 720360577]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'link-usd', datetime.datetime(2025, 12, 3, 0, 0), 13.51, 14.71, 13.46, 14.69, 14.69, 1211123454]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'link-usd', datetime.datetime(2025, 12, 2, 0, 0), 12.09, 13.57, 11.99, 13.51, 13.51, 799393506]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'link-usd', datetime.datetime(2025, 12, 1, 0, 0), 12.98, 12.98, 11.76, 12.09, 12.09, 829474677]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'link-usd', datetime.datetime(2025, 11, 30, 0, 0), 13.0, 13.41, 12.98, 12.98, 12.98, 328471524]
Saving to postgres db...done
test:  ['https://finance.yahoo.com', 'link-usd', datetime.datetime(2025, 11, 29, 0, 0), 13.14, 13.21, 12.96, 13.0, 13.0, 310763617]
Saving to postgres db...done